In [2]:
### code to walk thru images and create datasheet with paths and timestamps

# for each image
    # determine time stamp
    #
    
import os
from PIL import Image
from IPython.display import display
import easyocr

reader = easyocr.Reader(['en'])

image_directory = "E:/pukele_images/frames_1/TIMEL0001"

def extract_bottom_text_from_images(root_dir, r_ratio=0.2, bottom_ratio=0.2, show_crops=True):
    """
    Walk through all images in the given directory (recursively),
    crop the bottom part of each image, and print/display the detected text.

    Parameters:
        root_dir (str): Path to the root folder containing images.
        bottom_ratio (float): Fraction of image height to take from the bottom (default = 0.2).
        show_crops (bool): Whether to display the cropped bottom part (default True).
    """
    valid_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp')
    results = []  # store (path, text)

    for dirpath, _, filenames in os.walk(root_dir):
        for fname in filenames:
            if not fname.lower().endswith(valid_exts):
                continue

            path = os.path.join(dirpath, fname)
            try:
                img = Image.open(path)

                # Crop bottom portion of the image
                width, height = img.size
                crop_width = int(width * r_ratio)
                crop_height = int(height * bottom_ratio)
                bottom_region = img.crop((width - crop_width, height - crop_height, width, height))

                # Convert to grayscale for cleaner OCR
                bottom_region_gray = bottom_region.convert("L")

                # Extract text using pytesseract
                text = reader.readtext(bottom_region_gray, detail=0)
                text = pytesseract.image_to_string(bottom_region_gray).strip()
                results.append((path, text))

                # Display info and cropped region
                if show_crops:
                    print(f"\n📄 File: {path}")
                    print(f"Detected text: {text if text else '(no text detected)'}")
                    display(bottom_region)

            except Exception as e:
                print(f"\n❌ Error reading {path}: {e}")
                results.append((path, f"Error: {e}"))
                
    print(len(results))

    return results

results = extract_bottom_text_from_images(image_directory, r_ratio=0.25, bottom_ratio=0.042, show_crops=False)


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [19]:
import pandas as pd
from datetime import datetime

rows = []
for path, text in results:
    # Try to parse date/time if text looks like a timestamp
    try:
        dt = datetime.strptime(text, "%m/%d/%Y %I:%M%p")
        date_str = dt.strftime("%Y-%m-%d")
        time_str = dt.strftime("%H:%M")
    except Exception:
        # If it’s not in the expected format, leave blank
        date_str = ""
        time_str = ""
    
    rows.append({
        "date": date_str,
        "time": time_str,
        "raw_text": text,
        "path": path
    })

df = pd.DataFrame(rows)

# Save to Excel
output_path = "bottom_text_results.xlsx"
df.to_excel(output_path, index=False)

print(f"✅ Saved {len(df)} entries to {output_path}")
df.head()


✅ Saved 1685 entries to bottom_text_results.xlsx


,date,time,raw_text,path
0,,,,E:/pukele_images\frames_0\TIMEL0001\frame_0001...
1,,,,E:/pukele_images\frames_0\TIMEL0001\frame_0002...
2,,,,E:/pukele_images\frames_0\TIMEL0001\frame_0003...
3,,,,E:/pukele_images\frames_0\TIMEL0001\frame_0004...
4,,,,E:/pukele_images\frames_0\TIMEL0001\frame_0005...
